# 第17回　総合課題 — 手書き数字認識
***
> **前提**: 第16回で学習・保存した `best_mnist_model.pth` を使い，自作の手書き数字で推論します。
>
> **実行環境**: 手書き推論 UI は **Google Colab** 上での実行を推奨します（`google.colab.output` を使用）。Colab では第14〜16回を順に実行してから本課題に取り組んでください。ローカル Jupyter では問題1と問題3（考察）のみ実施可能です。

## 目次
1. モデルの読み込み
2. 手書き推論 UI（完成コード — 編集不要）
3. 推論結果の記録
4. 考察

---

## この回で学ぶこと

### 「学習データと推論データの分布ズレ」という根本的な問題

第14〜16回では MNIST データセットで学習し，MNIST のテストデータで評価してきた。しかし今回は「自分の手書き文字」という，学習データとは**異なる分布**のデータで推論する。

この問題を **Distribution Shift（分布シフト）** または **Domain Shift** と呼ぶ：

```
【学習時のデータ分布】         【推論時のデータ分布】
MNIST（白背景・黒文字・          自分の手書き（UI キャンバス：
  標準的なサイズ・               黒背景・白文字・
  中央揃え）                     個人差がある文字スタイル）
```

これが AI の実運用での最大の課題の一つだ。

### 前処理の一致がなぜ重要か

モデルは学習時の前処理を「前提」として学習している。推論時に異なる前処理をすると，モデルが意図しない入力を受け取ることになる：

```
学習時: PIL画像 → ToTensor() → [0,1]正規化 → 白地に黒文字
推論時（今回）: Canvas画像（黒地に白文字）→ 色反転 → リサイズ(28,28) → 同じ正規化
```

今回のコードで `arr = 1.0 - arr` （色反転）を行っているのは，この分布ズレを補正するためだ。

### 信頼度（Confidence）の意味

`torch.softmax(logits, dim=1)` の出力は各クラスの「確率」だ。最大値が「信頼度」として表示される。ただし：

- 信頼度が高い ≠ 必ず正解（モデルが確信を持って間違えることもある）
- これを「過信（Overconfidence）」と呼ぶ。深層学習モデルによく見られる問題だ
- 対策：Temperature Scaling, MC Dropout などの不確かさ推定手法がある

### 推論の高速化と本番デプロイ

今回の推論は1枚ずつ行うが，実際のサービスでは：
- バッチ推論（複数画像を一度に処理）
- `model.eval()` + `torch.no_grad()` は必須（速度・メモリ効率）
- ONNX Export（モデルを PyTorch 以外の環境でも動かす標準形式）
- TorchScript（Python 依存をなくして高速化）

卒業研究でも「作ったモデルを実際に動かす」ところまで取り組むと，研究の完成度が大きく上がる。

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1　モデルの読み込みとテスト精度の再確認
***

### なぜモデルを再読み込みして確認するのか

「保存したモデルを正しく読み込めているか」を確認することは，実際の運用でも重要なステップだ。保存と読み込みに成功していれば，テスト精度が第16回と全く同じ値になるはずだ。

### `load_state_dict` の使い方

```python
model = SimpleCNN().to(device)  # モデルの「構造」を定義（空箱を作る）
model.load_state_dict(          # 学習済みの「重み」を読み込む（中身を詰める）
    torch.load("best_mnist_model.pth", map_location=device)
)
model.eval()  # 推論モードに切り替え（必須！）
```

**`map_location=device` の意味**：GPU で保存したモデルを CPU 環境で読み込む場合（または逆）に，適切なデバイスに重みをマッピングする。これがないと GPU で保存したモデルが CPU 環境で読み込めないエラーが発生する。

### モデル構造の一致が必須

`SimpleCNN` クラスの定義が第16回と**完全に一致している必要がある**。層の数・サイズ・名前が一つでも違うと `load_state_dict` が失敗する。これは「モデルのバージョン管理」が重要な理由だ。

### 課題

第16回と同じ `SimpleCNN` クラスを定義し，`best_mnist_model.pth` から `load_state_dict` でモデルを復元してください。

テストデータでの正解率を再確認して出力し，第16回の結果と一致することを確認してください。

#### Hints
- `SimpleCNN` クラスは第16回で実装したものと**完全に同じ構造**でなければならない。構造が1つでも違うと `load_state_dict` が失敗する
- 重みの読み込みは `model.load_state_dict(torch.load(...))` の2ステップ。`map_location=device` を忘れると、GPU で保存したモデルをCPU環境で読み込めない場合がある
- 読み込み後は必ず `model.eval()` を呼ぶ
- 読み込みが成功したら、テスト正解率が第16回と同じ値になるはずなので確認して確かめる

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


# モデル読み込みとテスト正解率
# ここにあなたのコードを書いてください


## 手書き推論 UI
***
以下のセルは **完成コード** です。編集せず実行してください。

Canvas に 0〜9 の数字を書き，「推論する」ボタンを押すと予測結果が表示されます。

> **Canvas は黒背景・白線** です。MNIST は白背景・黒線のため，推論時に `1.0 - arr` で**色を反転**しています。これが Distribution Shift への対処法の一つだ。

### 推論の処理フロー（内部で何が起きているか）

```
1. Canvas からピクセルデータを取得（PNG → base64 → PIL Image）
2. グレースケールに変換（カラー情報は不要）
3. 28×28 にリサイズ（MNIST と同じサイズに）
4. 値を [0,1] に正規化（255で割る）
5. 色を反転（1.0 - arr）← Distribution Shift の補正
6. Tensor に変換して shape を (1, 1, 28, 28) に
7. model(tensor) で推論 → logits (1, 10)
8. softmax で確率に変換 → argmax で予測クラスを取得
```

In [ ]:
# 手書き推論UI（完成コード — 編集不要）
import numpy as np
from IPython.display import display, HTML
import base64
from PIL import Image
import io
from google.colab import output


def predict_digit(img_data_url):
    header, data = img_data_url.split(",", 1)
    img_bytes = base64.b64decode(data)
    img = Image.open(io.BytesIO(img_bytes)).convert("L")

    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype("float32") / 255.0
    arr = 1.0 - arr  # Canvas=黒地白線 → MNIST=白地黒線 に反転

    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        digit = int(probs.argmax(dim=1).item())
        confidence = float(probs.max().item()) * 100
    return digit, confidence


def _on_predict(data_url):
    digit, conf = predict_digit(data_url)
    print(f"予測: {digit}  信頼度: {conf:.1f}%")


output.register_callback("predict_from_js", _on_predict)

html_code = """
<style>
  body { font-family: sans-serif; }
  #canvas {
    border: 3px solid #333;
    border-radius: 8px;
    cursor: crosshair;
    background: black;
    display: block;
    margin: 10px 0;
  }
  .btn {
    padding: 10px 24px;
    margin: 4px;
    font-size: 16px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
  }
  #predictBtn { background: #4CAF50; color: white; }
  #clearBtn   { background: #f44336; color: white; }
  #result {
    font-size: 32px;
    font-weight: bold;
    margin-top: 12px;
    min-height: 40px;
    color: #1a73e8;
  }
  #confidence { font-size: 16px; color: #555; }
</style>

<h3>数字を書いてください（0〜9）</h3>
<canvas id="canvas" width="280" height="280"></canvas>

<div>
  <button class="btn" id="predictBtn" onclick="predict()">推論する</button>
  <button class="btn" id="clearBtn"   onclick="clearCanvas()">クリア</button>
</div>

<div id="result">ここに結果が表示されます</div>
<div id="confidence"></div>

<script>
  const canvas = document.getElementById("canvas");
  const ctx    = canvas.getContext("2d");

  ctx.fillStyle = "black";
  ctx.fillRect(0, 0, 280, 280);
  ctx.strokeStyle = "white";
  ctx.lineWidth   = 20;
  ctx.lineCap     = "round";
  ctx.lineJoin    = "round";

  let drawing = false;
  let lastX = 0, lastY = 0;

  function getPos(e) {
    const rect = canvas.getBoundingClientRect();
    if (e.touches) {
      return {
        x: e.touches[0].clientX - rect.left,
        y: e.touches[0].clientY - rect.top
      };
    }
    return { x: e.clientX - rect.left, y: e.clientY - rect.top };
  }

  canvas.addEventListener("mousedown",  e => { drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("mousemove",  e => {
    if (!drawing) return;
    const p = getPos(e);
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(p.x, p.y);
    ctx.stroke();
    lastX = p.x; lastY = p.y;
  });
  canvas.addEventListener("mouseup",   () => drawing = false);
  canvas.addEventListener("mouseleave",() => drawing = false);

  canvas.addEventListener("touchstart",  e => { e.preventDefault(); drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchmove",   e => { e.preventDefault(); if (!drawing) return; const p = getPos(e); ctx.beginPath(); ctx.moveTo(lastX, lastY); ctx.lineTo(p.x, p.y); ctx.stroke(); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchend",    e => { e.preventDefault(); drawing = false; });

  function clearCanvas() {
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, 280, 280);
    document.getElementById("result").innerText = "ここに結果が表示されます";
    document.getElementById("confidence").innerText = "";
  }

  function predict() {
    const dataURL = canvas.toDataURL("image/png");
    google.colab.kernel.invokeFunction("predict_from_js", [dataURL], {});
  }
</script>
"""

display(HTML(html_code))
print("キャンバスを表示しました。数字を書いて「推論する」を押してください。")


## 問題2　推論結果の記録
***

### 記録すべき情報

UI で0〜9の数字を書いて推論し，以下の情報を記録してください：
- **正解ラベル**: 書いた数字
- **予測**: モデルが予測した数字
- **信頼度（%）**: モデルの確信度

### 誤認識パターンの分析

誤認識が起きた場合，以下を確認しよう：
1. **信頼度は高いか低いか**: 低信頼度（60%以下）の誤認識は「曖昧な入力」が原因
2. **どの数字に間違えたか**: 第16回の混同行列と一致するパターンか
3. **書き方は普通か**: 独特な書き方（例：欧風の「1」に横棒を付けるなど）が原因なことも

### Distribution Shift の観察

自分の手書き文字の精度が，第16回のテストデータ（MNIST）の精度より低い場合，それは Distribution Shift の影響だ。なぜ低くなるかを考えてみよう：
- 文字の太さの違い
- 文字の傾き
- キャンバス内での位置（MNIST は中央揃え）

### 課題

UI で **0〜9 を各1回以上** 書き，予測結果を表形式（DataFrame）で記録してください。

正解率（自分で書いた数字のうち正しく認識された割合）を計算して出力してください。

#### Hints
- UI から得た結果を辞書のリストとして手入力し、`pd.DataFrame` に渡すとテーブルになる
- 正解率は「正解列と予測列が一致する行の割合」で計算できる。比較演算子と `.mean()` を組み合わせる
- 誤認識した数字があった場合は、その行にコメントを追加して記録しておくと考察に使いやすい

In [ ]:
# 推論結果の記録
# ここにあなたのコードを書いてください


## 問題3　総合考察
***

### この問いに答えることの意味

単に「コードを動かした」だけでなく，**なぜそうなったか** を自分の言葉で説明できることが，真の理解の証明だ。卒業研究では実験結果の考察は必須であり，「モデルが良い/悪かった → なぜか → どうすれば改善できるか」という思考の流れを練習しよう。

### 考察のための視点

**① CNN が MLP より精度が高い理由について**：
- 畳み込み層は「局所的な特徴」を検出する。数字の「丸み」「直線」「交差点」など，28×28 の中の小さなパターンを見つけられる
- 重みを画像全体で共有（重み共有）するため，パラメータが少ない割に表現力が高い
- MLP は784個のピクセルをバラバラに扱うため，「隣接するピクセルの関係性」を学習しにくい

**② 自作数字の誤認識原因について**：
- MNIST は特定のスタイルの手書き文字だけで学習している
- 自分の書き方がトレーニングデータの分布から外れると精度が下がる
- 「前処理（リサイズ，色反転，正規化）が正しく行われているか」も確認が必要

**③ 精度向上のアイデアについて（選択肢）**：
- **Data Augmentation（データ拡張）**: 学習データを回転・拡大・ノイズ追加などで増やす → テストへの汎化性が上がる
- **Dropout**: 学習時に一部のニューロンをランダムに無効化 → 過学習を防ぐ
- **Batch Normalization**: 各層の出力を正規化 → 学習が安定・高速化
- **エポック数を増やす**: 損失曲線を見て，まだ収束していないなら有効
- **より深い CNN**: VGG, ResNet など実績のあるアーキテクチャを使う

### 課題

以下の3点について，**print または markdown セル**を使って自分の言葉で述べてください（各2〜4文程度）：

1. **第16回で CNN が MLP（LinearNet, DeepMLP）より精度が高かった理由**を，CNN の仕組みを踏まえて説明してください
2. **自作数字で誤認識が起きた場合（または起きなかった場合）の原因**を，Distribution Shift や前処理の観点から考察してください
3. **精度をさらに上げるために試せる手法を2つ**挙げ，それぞれ「なぜ効果があると思うか」を説明してください

> **ポイント**: 「正解率が高かった/低かった」という事実だけでなく，「なぜそうなったか」「どう改善できるか」の考察が重要だ。


In [ ]:
# 考察
# ここにあなたのコードを書いてください
